# c1 — Reported drift cells (LOCAL-REPO TEST VERSION)

**This version reads everything from the local repository, not Google Drive.** Run it with a kernel on this laptop
(VS Code / Jupyter using `C:\lexsi\.venv`, or Colab → *Connect to a local runtime*).

| What | Where it comes from now |
|---|---|
| code (harness, merge_stream, medmcqa_score) | `<repo>/harness`, `<repo>/finetune`, `<repo>/scripts/eval` |
| prompts | `<repo>/Datasets/final/<set>/prompts.jsonl` |
| adapters | `<repo>/models/runs/{med,ctrl}/adapter` |
| reference tensor hashes | `<repo>/models/merged/{med,ctrl}/tensor_sha256.json` |
| merged models | `<repo>/models/merged/{med,ctrl}` (or rebuilt into `models/merged_rebuild/` if `REBUILD = True`) |
| HF cache | `<repo>/models/hf_cache` |
| HF read token | `HF_TOKEN_READ` in `<repo>/.env` (falls back to a prompt) |
| outputs | `<repo>/results/local_test/{private,cells}/<cell>/` |

Outputs go to `results/local_test/`, **not** `results/cells/`, so a laptop test is never mistaken for the reported T4 cells.
Set `LIMIT` in step 4 to a small number (e.g. 32) for a quick smoke test; `LIMIT = 0` runs all 1,612 prompts per cell.
Everything is resumable. To get the Google Drive version back: `python scripts/make_colab_notebooks.py`.


In [ ]:
!pip install bitsandbytes

In [3]:
# 1. GPU check + versions (the local venv already has the laptop's pins; nothing is installed here)
import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)
assert torch.cuda.is_available(), "No GPU visible to this kernel"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
PINS = {"transformers": "5.22.1", "peft": "0.21.2", "bitsandbytes": "0.50.2"}
for k, v in PINS.items():
    have = __import__(k).__version__
    if have != v:
        print(f"WARNING: {k} {have} != pinned {v}")


torch 2.11.0+cu130 | cuda 13.0 | transformers 5.22.0 | peft 0.21.2 | bnb 0.50.2
Tesla T4 15.6 GB


In [5]:
# 2. Local repo paths (code + prompts read in place: no zips, no Drive)
import os, json, time, subprocess, sys
from pathlib import Path
_cands = [Path(os.environ.get("LEXSI_REPO", "")), Path.cwd().parent, Path.cwd(),
          Path(r"C:\Users\SUSHOVIT\OneDrive\Desktop\Lexsi_Hackathon")]
REPO = next(c for c in _cands if str(c) not in ("", ".") and (c / "harness" / "run_cell.py").exists())
WORK = str(REPO)
OUT = str(REPO / "results" / "local_test"); os.makedirs(OUT, exist_ok=True)
os.environ["HF_HOME"] = str(REPO / "models" / "hf_cache"); os.environ["PYTHONUTF8"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
CELL_SETS = ["aya_en_test", "aya_hi_test", "aya_hinglish_test", "clinicsafe_hi_en", "clinicsafe_hi_hi", "clinicsafe_hi_hinglish",
             "xstest_en_test", "xstest_hi_test", "xstest_hinglish_test", "medmcqa_test_500"]
n = {s: sum(1 for _ in open(REPO / "Datasets" / "final" / s / "prompts.jsonl", encoding="utf-8")) for s in CELL_SETS}
print("REPO:", REPO); print("OUT :", OUT)
print({"cell_sets": n, "generations_per_cell": sum(v for k, v in n.items() if k != "medmcqa_test_500")})


StopIteration: 

In [ ]:
# 3. Hugging Face READ token from the repo's .env (never printed); prompts only if it is missing
from huggingface_hub import login
tok = None
envf = REPO / ".env"
if envf.exists():
    for line in envf.read_text(encoding="utf-8").splitlines():
        if line.strip().startswith("HF_TOKEN_READ="):
            tok = line.split("=", 1)[1].strip().strip('"').strip("'")
if not tok:
    from getpass import getpass
    tok = getpass("HF read token: ")
login(token=tok); del tok
print("logged in")


In [ ]:
# 4. Choose this session's cells (edit this list)
ALL_CELLS = [
 "orig__fp16__f16__tpl-official",
 "orig__nf4__f16__tpl-official",
 "med__fp16__f16__tpl-official",
 "med__nf4__f16__tpl-official",
 "ctrl__fp16__f16__tpl-official",
 "ctrl__nf4__f16__tpl-official"
]
RUN_CELLS = ALL_CELLS[:3]      # session 1: orig FP16, orig NF4, med FP16   |   session 2: ALL_CELLS[3:]
SETS = ["aya_en_test", "aya_hi_test", "aya_hinglish_test", "clinicsafe_hi_en", "clinicsafe_hi_hi", "clinicsafe_hi_hinglish", "xstest_en_test", "xstest_hi_test", "xstest_hinglish_test"]
MODEL_OF = {c: c.split("__")[0] for c in ALL_CELLS}
WEIGHTS_OF = {c: c.split("__")[1] for c in ALL_CELLS}
LIMIT = 0                      # >0 = quick smoke test: only the first LIMIT prompts per cell (MedMCQA skipped)
print("this session:", RUN_CELLS, "| LIMIT:", LIMIT or "all")


In [ ]:
# 5. Merged models from the local repo. REBUILD=True re-merges from the adapters (streamed) into
#    models/merged_rebuild/ and checks every tensor hash against models/merged/<m>/tensor_sha256.json.
REBUILD = False
need = sorted({MODEL_OF[c] for c in RUN_CELLS} - {"orig"})
MODEL_PATH = {"orig": "CohereLabs/tiny-aya-global"}
for m in need:
    ref_dir = REPO / "models" / "merged" / m
    ref = json.load(open(ref_dir / "tensor_sha256.json"))["tensor_sha256"]
    if not REBUILD:
        assert (ref_dir / "model.safetensors.index.json").exists(), f"{ref_dir} missing: set REBUILD = True"
        MODEL_PATH[m] = str(ref_dir); print(f"{m}: using {ref_dir}"); continue
    out = REPO / "models" / "merged_rebuild" / m
    if not (out / "tensor_sha256.json").exists():
        r = subprocess.run([sys.executable, str(REPO / "finetune" / "merge_stream.py"), "--adapter", str(REPO / "models" / "runs" / m / "adapter"),
                            "--out", str(out)], capture_output=True, text=True, encoding="utf-8", errors="replace", cwd=WORK)
        print(r.stdout[-600:], r.stderr[-1500:])
        assert r.returncode == 0, "merge failed"
    mine = json.load(open(out / "tensor_sha256.json"))["tensor_sha256"]
    bad = [k for k in ref if mine.get(k) != ref[k]]
    print(f"{m}: {len(ref) - len(bad)}/{len(ref)} tensors identical to models/merged/{m}")
    assert not bad, f"HASH MISMATCH in {len(bad)} tensors (e.g. {bad[:3]})"
    MODEL_PATH[m] = str(out)
print("models ready:", MODEL_PATH)


In [ ]:
# 6. Generate each cell (resumable) + MedMCQA test-500 one-token accuracy (full runs only)
t_session = time.time()
for cell in RUN_CELLS:
    m, w = MODEL_OF[cell], WEIGHTS_OF[cell]
    print(f"\n===== {cell}  (model={m}, weights={w}) =====", flush=True)
    cmd = [sys.executable, str(REPO / "harness" / "run_cell.py"), "--cell", cell, "--model", MODEL_PATH[m], "--weights", w,
           "--sets", *SETS, "--out", OUT, "--batch", "16"] + (["--limit", str(LIMIT)] if LIMIT else [])
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", cwd=WORK)
    for line in p.stdout:
        if line.startswith("[") or "OOM" in line or "Error" in line or "Traceback" in line:
            print(line, end="", flush=True)
    assert p.wait() == 0, f"{cell} failed: rerun this cell to resume"
    mq = f"{OUT}/cells/{cell}/medmcqa_test500.json"
    if not os.path.exists(mq) and not LIMIT:
        cmd = [sys.executable, str(REPO / "scripts" / "eval" / "medmcqa_score.py"), "--model", MODEL_PATH[m],
               "--set", str(REPO / "Datasets" / "final" / "medmcqa_test_500" / "prompts.jsonl"), "--out", mq] + (["--nf4"] if w == "nf4" else [])
        r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace", cwd=WORK)
        print([l for l in r.stdout.splitlines() if "accuracy" in l] or r.stderr[-800:])
    print(f"session elapsed {(time.time() - t_session) / 3600:.2f} h", flush=True)


In [ ]:
# 7. Summary block: copy everything between the markers back into the chat
summary = {}
for cell in ALL_CELLS:
    rec = f"{OUT}/cells/{cell}/records.jsonl"
    info = f"{OUT}/cells/{cell}/run_info.json"
    mq = f"{OUT}/cells/{cell}/medmcqa_test500.json"
    s = {"generated": sum(1 for _ in open(rec, encoding="utf-8")) if os.path.exists(rec) else 0, "expected": LIMIT or 1612}
    if os.path.exists(info): s.update({k: v for k, v in json.load(open(info)).items() if k in ("gen_per_min", "tok_per_s", "peak_gb", "seconds", "engine")})
    if os.path.exists(mq): s["medmcqa_test500"] = round(json.load(open(mq))["accuracy"], 4)
    summary[cell] = s
print("=====C1_RESULTS_BEGIN====="); print(json.dumps(summary, indent=1)); print("=====C1_RESULTS_END=====")
print(f"\nOutputs are in {OUT}  (local test only; the reported cells come from the T4 run).")
